# S3 · Library of Babel scale, computed
The premise cited under the paper's title: Basile, libraryofbabel.info, About page: the site holds all pages of 3,200 characters ("approximately 10^4677 books"); the complete library would hold every combination of 1,312,000 characters. Below, those numbers are recomputed and the exact shipped class is run on a 29-symbol charset. The site's own numbering algorithm is not examined here; the numbering below is this paper's.

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
LOG10_29 = math.log10(29)
page_chars, book_chars, pages_per_book = 3200, 1_312_000, 410
log_pages = page_chars * LOG10_29
print("pages of 3,200 characters over 29 symbols: 10^%.2f" % log_pages)
print("divided by 410 pages per book:            10^%.2f  (the site says ~10^4677)" % (log_pages - math.log10(pages_per_book)))
print("books of 1,312,000 characters:            10^%.0f" % (book_chars * LOG10_29))
print("1,312,000 = 410 x 3,200 =", pages_per_book * page_chars, "; = 410 pages x 40 lines x 80 chars =", 410 * 40 * 80)
assert round(log_pages - math.log10(pages_per_book)) == 4677

pages of 3,200 characters over 29 symbols: 10^4679.67
divided by 410 pages per book:            10^4677.06  (the site says ~10^4677)
books of 1,312,000 characters:            10^1918666
1,312,000 = 410 x 3,200 = 1312000 ; = 410 pages x 40 lines x 80 chars = 1312000


## The shipped class on the Babel alphabet
29 symbols: 26 letters, space, comma, period. A page is a 3,200-character string; its address is a base-29 numeral.

In [3]:
import random
hw2 = load("PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hyperwebster.py", "hw2")
B = hw2.HyperWebster("abcdefghijklmnopqrstuvwxyz ,.")
print("N =", B.N)
rnd = random.Random(1)
page = "".join(rnd.choice(B.characters) for _ in range(page_chars))
t0 = time.perf_counter(); a = B.point_to_text(page); t1 = time.perf_counter(); back = B.regenerate_text(a); t2 = time.perf_counter()
assert back == page
print("page of %d chars -> address of %d bits (= %.1f bits/char; log2(29) = %.3f) | encode %.0f ms, decode %.0f ms" % (page_chars, a.bit_length(), a.bit_length()/page_chars, math.log2(29), 1000*(t1-t0), 1000*(t2-t1)))
print("relative to 8-bit text: %.3f" % (a.bit_length() / (8 * page_chars)), "= log2(29)/8 =", round(math.log2(29)/8, 3))

N = 29
page of 3200 chars -> address of 15544 bits (= 4.9 bits/char; log2(29) = 4.858) | encode 264 ms, decode 33 ms
relative to 8-bit text: 0.607 = log2(29)/8 = 0.607


## A page's location is one integer: fixed-length pages and shortlex ranks
Among *all* strings the class ranks by length first; among the 3,200-character pages the location is the plain base-29 numeral of the page. The offset between the two numberings is the count of shorter strings, (29³²⁰⁰ − 29)/28.

In [4]:
plain = 0
for ch in page: plain = plain * 29 + B._char_index[ch]
shorter = (29 ** page_chars - 29) // 28
print("shortlex address - number of shorter strings == plain base-29 value:", a - shorter == plain); assert a - shorter == plain
print("first page (3,200 x 'a') has location 0; last page (3,200 x '.') has location", 29**page_chars - 1 == B.point_to_text("." * page_chars) - shorter)

shortlex address - number of shorter strings == plain base-29 value: True
first page (3,200 x 'a') has location 0; last page (3,200 x '.') has location True


## Neighbouring pages are neighbouring integers
Changing only the last character by one step in the charset order moves the location by exactly 1; changing the first moves it by 29³¹⁹⁹.

In [5]:
chars = B.characters
i = chars.index(page[-1]); j = (i + 1) % 29
p2 = page[:-1] + chars[j]
if j > i: print("last character +1 step -> location difference:", B.point_to_text(p2) - a); assert B.point_to_text(p2) - a == 1
q = chars.index(page[0]); r = (q + 1) % 29
if r > q:
    p3 = chars[r] + page[1:]; print("first character +1 step -> location difference == 29^3199:", B.point_to_text(p3) - a == 29 ** 3199)
else: print("first character wraps; skipped")

last character +1 step -> location difference: 1
first character +1 step -> location difference == 29^3199: True


## One book: size is exact, time is an extrapolation (stated as such)
A 1,312,000-character book has an address of 1,312,000 · log₂29 bits. Encoding time is measured at four sizes, the exponent is fitted, and the time for a book is extrapolated from the fit.

In [6]:
sizes = [2000, 4000, 8000, 16000]; ts = []; tf = []
def forward(t, N=29, idx=B._char_index):
    a = 0
    for c in t: a = a * N + idx[c] + 1
    return a - 1
for s in sizes:
    txt = "".join(rnd.choice(B.characters) for _ in range(s))
    t0 = time.perf_counter(); x = B.point_to_text(txt); t1 = time.perf_counter(); y = forward(txt); t2 = time.perf_counter()
    assert x == y; ts.append(t1 - t0); tf.append(t2 - t1)
import math as m
fit = lambda v: m.log(v[-1] / v[0]) / m.log(sizes[-1] / sizes[0])
book_bits = book_chars * m.log2(29)
print("chars  point_to_text s  forward Horner s")
for s, a, b in zip(sizes, ts, tf): print("%5d  %15.3f  %16.4f" % (s, a, b))
print("fitted exponents: point_to_text %.2f | forward Horner %.2f" % (fit(ts), fit(tf)))
print("book address: %.0f bits = %.0f KiB" % (book_bits, book_bits / 8 / 1024))
for name, v in (("point_to_text", ts), ("forward Horner", tf)):
    est = v[-1] * (book_chars / sizes[-1]) ** fit(v)
    print("ESTIMATE, not measured: encoding one book with %-14s ~ %10.0f s (%.1f h)" % (name, est, est / 3600))

chars  point_to_text s  forward Horner s
 2000            0.028            0.0020
 4000            0.163            0.0072
 8000            1.009            0.0214
16000            6.074            0.0809
fitted exponents: point_to_text 2.59 | forward Horner 1.78
book address: 6373671 bits = 778 KiB
ESTIMATE, not measured: encoding one book with point_to_text  ~     542739 s (150.8 h)
ESTIMATE, not measured: encoding one book with forward Horner ~        203 s (0.1 h)
